# Kapitel 5 - Dimensionsreducering

## Faktafrågor

<font color='orange'> 1. Vad menas med curse of dimensionality. </font>

Svar: det är att i högre dimensioner så kan data vara på ett sätt som vi inte förväntat oss eller är vana vid från lägre dimensioner. ett exempel är då att om man slumpmässigt sätter ut en prick i en ruta så är fen en låg chans att den hamnar i ett hörn, men om vi då ökar dimensionen till 1000 så är där många fler hörn och en ökad sannolikhet

<font color='orange'> 2. Vad är dimensionsreducering och varför görs det? </font>

Svar: det är att vi transformerar ett dataset med en given dimension till ett lägre antal dimensioner. Detta görs för att minska antalet variabler

<font color='orange'> 3. Förklara översiktligt hur PCA fungerar. Använd figur 5.4 på sidan 224 i
din förklaring. </font>

Svar:
PCA hittar de axlar som bevarar mest varians när datan projiceras ner på dem. I figur 5.4 ser vi tre möjliga linjer att projicera ett 2D-dataset på; till höger syns hur spridd (z1) datan blir för varje linje. Den heldragna linjen, c1, ger störst spridning och blir första principalkomponenten, den prickade ger minst spridning och tappar mest information.

c2 är vinkelrät mot c1 och fångar mest av den varians som blir kvar. Genom att bara behålla de första komponenterna (tex c1) kan man reducera antalet dimensioner och ändå behålla så mycket information som möjligt.

<font color='orange'> 4. Hur kan kernel PCA utvärderas? </font>

Svar:

## Resonemangfrågor

<font color='orange'> 5. Stina påstår att man i maskininlärning alltid vill ha modeller som genomför
så bra prediktioner som möjligt. Kalle påstår att det inte riktigt stämmer
eftersom tid också är en viktig aspekt. Både för själva modellträningen
och för själva prediktionerna. Vad säger du? </font>

Svar:
Kalle har en poäng. I praktiken finns det ofta en avvägning mellan hur bra en modell presterar och hur lång tid tränings/prediktion tar. En något sämre men mycket snabbare modell kan vara att föredra, tex vid realtidsapplikationer (självkörande bilar, mobilappar) där prediktionstiden är avgörande, eller när man har begränsad tid/resurser för träning och behöver träna om modellen ofta. Dimensionsreducering med PCA är ett bra exempel, man tappar lite information men vinner ofta mycket i tränings- och prediktionstid, vilket kan vara värt det beroende på vad modellen ska användas till.

<font color='orange'> 6. Efter att vi genomfört en PCA, vad händer med tolkningen av variablerna? </font>

Svar:
De nya variablerna (principalkomponenterna) är linjärkombinationer av de ursprungliga variablerna, och tappar därför sin direkta, verkliga betydelse. Man kan inte längre säga att en komponent motsvarar tex "ålder" eller "pris", utan varje komponent är en blandning av flera ursprungliga variabler viktade olika mycket. Det blir alltså svårare att tolka vad en hög eller låg komponent faktiskt innebär, man förlorar tolkbarhet i utbyte mot färre dimensioner.

## Koduppgifter

<font color='orange'> 7. Gå igenom samtliga kodexempel i kapitlet och skriv gärna av koden ma-
nuellt. Det är även bra att experimentera genom att ändra vissa delar av
koden och läsa dokumentationen. </font>

Svar:

<font color='orange'> 8. Förklara vad nedanstående kod gör.
```
import numpy as np
from sklearn.decomposition import PCA
# Creating a dataset with 3 features/columns
X = np.random.rand(1000, 3)
print(X[0:5])
# Reducing the data to 2 dimensions
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])
# "Recreating" the data to 3 dimensions
X3D_inv = pca.inverse_transform(X2D)
# Not exactly equal since some information was lost in the
transformation↪
print(np.allclose(X3D_inv, X)) 
```

</font>

Svar:
Koden skapar ett slumpmässigt dataset X med 1000 rader och 3 features (dimensioner). Sedan används PCA för att reducera datan till 2 dimensioner (X2D) via fit_transform(), där PCA:n både lär sig principalkomponenterna från datan och projicerar ner den. Därefter används inverse_transform() för att försöka återskapa den ursprungliga 3D-datan (X3D_inv) utifrån de 2 dimensionerna.

Resultatet av np.allclose(X3D_inv, X) blir False, eftersom man vid dimensionsreduceringen (3D till 2D) tappade information (variansen som fanns i den tredje, bortplockade riktningen), och den informationen går inte att återskapa exakt. X3D_inv blir alltså en approximation av originaldatan, inte en identisk kopia.

<font color='orange'> 9. Genomför en PCA på “car_price_dataset.csv” från kapitel 3 innan du
modellerar det med ML. Hur påverkas resultatet? </font>

Svar:
Jag återanvänder samma preprocessing och modeller som i uppgift 15 i kapitel 3 (one hot encoding av kategoriska variabler, skalning av numeriska), och jämför sedan resultatet med och utan PCA innan modelleringen.

In [1]:
import pandas as pd

df = pd.read_csv('dataset/car_price_dataset.csv', sep=';')
df.head()

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

categorical_features = ['Brand', 'Model', 'Fuel_Type', 'Transmission']
numerical_features = ['Year', 'Engine_Size', 'Mileage', 'Doors', 'Owner_Count']

X = df.drop(columns='Price')
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('num', StandardScaler(), numerical_features),
])

n_dims = preprocessor.fit_transform(X_train).shape[1]
print(f'Antal dimensioner efter preprocessing (one hot + scaling): {n_dims}')

Antal dimensioner efter preprocessing (one hot + scaling): 52


Först tränas en baseline utan PCA, precis som i kapitel 3.

In [3]:
def evaluate(pipeline, name):
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)
    print(f'{name} -> RMSE: {rmse:.2f}, R2: {r2:.4f}')
    return rmse, r2

lin_pipeline = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
rf_pipeline = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))])

evaluate(lin_pipeline, 'Linjär regression')
evaluate(rf_pipeline, 'Random forest')

Linjär regression -> RMSE: 64.91, R2: 0.9995
Random forest -> RMSE: 327.93, R2: 0.9883


(327.9319372941756, 0.9882954334211247)

Sedan läggs ett PCA-steg in i pipelinen efter preprocessing, med n_components=0.95 så att PCA:n behåller det antal komponenter som krävs för att bevara 95 % av variansen.

In [4]:
from sklearn.decomposition import PCA

lin_pca = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95, svd_solver='full', random_state=42)),
    ('model', LinearRegression()),
])
rf_pca = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95, svd_solver='full', random_state=42)),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])

evaluate(lin_pca, 'Linjär regression + PCA')
evaluate(rf_pca, 'Random forest + PCA')
print(f'Antal principalkomponenter (95% av variansen): {lin_pca.named_steps["pca"].n_components_} av {n_dims} ursprungliga dimensioner')

Linjär regression + PCA -> RMSE: 64.99, R2: 0.9995
Random forest + PCA -> RMSE: 576.95, R2: 0.9638
Antal principalkomponenter (95% av variansen): 27 av 52 ursprungliga dimensioner


Hur resultatet påverkas: PCA reducerar antalet dimensioner från 52 till 27 (fortfarande 95 % av variansen bevarad), men effekten på modellerna skiljer sig kraftigt åt. Den linjära regressionen påverkas i princip inte alls (RMSE/R2 nästan identiska), eftersom PCA bara är en linjär omskrivning av samma information och en linjär modell kan lära sig lika bra av de nya komponenterna. Random forest däremot blir klart sämre (RMSE nästan dubbleras, R2 sjunker från 0.988 till 0.964), eftersom trädbaserade modeller gynnas av att kunna dela upp datan på enskilda, tydliga features (tex en specifik bilmärke-kolumn från one hot encodingen), vilket blir svårare när informationen är blandad över flera principalkomponenter. Slutsatsen är att PCA här ger snabbare träning på färre dimensioner, men på bekostnad av prestanda för trädmodeller, medan linjära modeller är mer okänsliga för PCA.